<a href="https://colab.research.google.com/github/rafiebazine/rag-image-captioning/blob/hadil/phase_02_preprocessing/02_metadata_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1: Setup

In [1]:
!git clone https://github.com/rafiebazine/rag-image-captioning.git
%cd rag-image-captioning
!git checkout hadil
!git merge origin/hadil

# COCO annotations (instances only)
!wget -q --show-progress http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -q -o annotations_trainval2017.zip "annotations/instances_train2017.json" "annotations/instances_val2017.json"

Cloning into 'rag-image-captioning'...
remote: Enumerating objects: 171, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 171 (delta 9), reused 4 (delta 2), pack-reused 133 (from 1)
Receiving objects: 100% (171/171), 2.72 MiB | 6.10 MiB/s, done.
Resolving deltas: 100% (67/67), done.
/content/rag-image-captioning
branch 'hadil' set up to track 'origin/hadil'.
Switched to a new branch 'hadil'
Already up to date.
annotations_trainva 100%[===================>] 241.19M  96.5MB/s    in 2.5s    


In [2]:
import json, os
from collections import defaultdict, Counter

SPLIT_DIR = "phase_01_data_exploration/splitted_data_set"
OUT_DIR = "phase_02_preprocessing/processed"
os.makedirs(OUT_DIR, exist_ok=True)

split_names = ["kb_10k", "val_100", "test_100", "test_1000"]
splits = {n: json.load(open(f"{SPLIT_DIR}/{n}.json")) for n in split_names}
needed_ids = {x["image_id"] for d in splits.values() for x in d}
print("images needed:", len(needed_ids))

metadata = {}   # image_id -> dict

for ann_file in ["annotations/instances_train2017.json", "annotations/instances_val2017.json"]:
    coco = json.load(open(ann_file))
    cats = {c["id"]: (c["name"], c["supercategory"]) for c in coco["categories"]}
    objs = defaultdict(list)
    for a in coco["annotations"]:
        if a["image_id"] in needed_ids:
            name, sup = cats[a["category_id"]]
            objs[a["image_id"]].append({
                "category": name, "supercategory": sup,
                "bbox": [round(v, 1) for v in a["bbox"]],   # [x, y, w, h]
                "area": round(a["area"], 1), "iscrowd": a["iscrowd"],
            })
    for img in coco["images"]:
        iid = img["id"]
        if iid in needed_ids:
            o = objs.get(iid, [])
            metadata[iid] = {
                "image_id": iid,
                "width": img["width"], "height": img["height"],
                "objects": o,
                "object_counts": dict(Counter(x["category"] for x in o)),
                "num_objects": len(o),
                "num_categories": len({x["category"] for x in o}),
                "supercategories": sorted({x["supercategory"] for x in o}),
                "source": "COCO instances ground-truth annotations (no detector, confidence N/A)",
            }
    del coco, objs
    print(ann_file, "done")

print("found:", len(metadata), "/", len(needed_ids))

images needed: 11200
annotations/instances_train2017.json done
annotations/instances_val2017.json done
found: 11200 / 11200


In [3]:
assert len(metadata) == len(needed_ids), "some images missing metadata"

for n, data in splits.items():
    ms = [metadata[x["image_id"]] for x in data]
    no_obj = sum(m["num_objects"] == 0 for m in ms)
    avg = sum(m["num_objects"] for m in ms) / len(ms)
    print(f"{n}: {len(ms)} images | avg objects {avg:.1f} | images with 0 objects: {no_obj}")
    json.dump(ms, open(f"{OUT_DIR}/metadata_{n}.json", "w"))
print("saved")


kb_10k: 10000 images | avg objects 6.7 | images with 0 objects: 219
val_100: 100 images | avg objects 7.5 | images with 0 objects: 2
test_100: 100 images | avg objects 5.4 | images with 0 objects: 0
test_1000: 1000 images | avg objects 5.7 | images with 0 objects: 0
saved
